In [1]:
# IMPORT LIBRARIES
import pandas as pd
import re
from collections import defaultdict, Counter
from google.colab import drive

# MOUNT GOOGLE DRIVE
drive.mount('/content/drive')

# LOAD DATASET
path = "/content/drive/MyDrive/autofill/movie_lines.tsv"

df = pd.read_csv(path, header=None, nrows=200000)

dialogues = df[0].astype(str).str.split("\t").str[-1]

# CLEAN TEXT
text = " ".join(dialogues)
text = text.lower()
text = re.sub(r'[^a-zA-Z\s]', '', text)

words = text.split()

print("Total words:", len(words))

# BUILD MARKOV CHAINS
tri_chain = defaultdict(list)
bi_chain = defaultdict(list)
uni_chain = defaultdict(list)

for i in range(len(words)-3):

    # trigram
    tri_key = (words[i], words[i+1], words[i+2])
    tri_chain[tri_key].append(words[i+3])

    # bigram
    bi_key = (words[i+1], words[i+2])
    bi_chain[bi_key].append(words[i+3])

    # unigram
    uni_key = (words[i+2],)
    uni_chain[uni_key].append(words[i+3])

print("Trigram states:", len(tri_chain))
print("Bigram states:", len(bi_chain))
print("Unigram states:", len(uni_chain))


# AUTOCOMPLETE FUNCTION
def predict_suggestions(sentence):

    sentence = sentence.lower()
    sentence = re.sub(r'[^a-zA-Z\s]', '', sentence)

    words_input = sentence.split()

    if len(words_input) == 0:
        return ["Type something"]

    # TRY TRIGRAM
    if len(words_input) >= 3:
        key = tuple(words_input[-3:])
        if key in tri_chain:
            common = Counter(tri_chain[key]).most_common(5)
            return [sentence + " " + w for w,_ in common]

    # TRY BIGRAM
    if len(words_input) >= 2:
        key = tuple(words_input[-2:])
        if key in bi_chain:
            common = Counter(bi_chain[key]).most_common(5)
            return [sentence + " " + w for w,_ in common]

    # TRY UNIGRAM
    key = (words_input[-1],)
    if key in uni_chain:
        common = Counter(uni_chain[key]).most_common(5)
        return [sentence + " " + w for w,_ in common]

    # FINAL FALLBACK
    common_words = Counter(words).most_common(5)
    return [sentence + " " + w for w,_ in common_words]


# INTERACTIVE AUTOCOMPLETE
while True:

    user_input = input("Enter sentence: ")

    results = predict_suggestions(user_input)

    print("Suggestions:")
    for r in results:
        print("-", r)

Mounted at /content/drive
Total words: 2063774
Trigram states: 1378250
Bigram states: 569093
Unigram states: 49398
Enter sentence: how to
Suggestions:
- how to get
- how to do
- how to make
- how to play
- how to handle
Enter sentence: draw
Suggestions:
- draw the
- draw up
- draw you
- draw it
- draw on
Enter sentence: the yellow  bench
Suggestions:
- the yellow  bench in
- the yellow  bench presses
- the yellow  bench behind
- the yellow  bench do
- the yellow  bench the


KeyboardInterrupt: Interrupted by user

In [ ]:
import numpy as np

# Number of pages
n = 4

# Adjacency matrix (links between pages)
# If page i links to page j → 1
A = np.array([
    [0, 1, 1, 0],  # Page A links to B and C
    [0, 0, 1, 0],  # Page B links to C
    [1, 0, 0, 1],  # Page C links to A and D
    [0, 0, 1, 0]   # Page D links to C
])

# Convert to transition probability matrix
M = A / A.sum(axis=1, keepdims=True)

# PageRank parameters
d = 0.85
# damping factor
iterations = 100

# Initial rank
rank = np.ones(n) / n

# Iterative computation
for i in range(iterations):
    rank = (1 - d) / n + d * M.T.dot(rank)

# Display result
for i, score in enumerate(rank):
    print(f"Page {chr(65+i)} Rank: {score:.4f}")

Page A Rank: 0.2199
Page B Rank: 0.1310
Page C Rank: 0.4292
Page D Rank: 0.2199
